# Spurious Correlation Sample Extractor

This notebook extracts samples from MedQA and MedXpertQA datasets that are relevant to specific spurious correlations.

**Workflow:**
1. Define a spurious correlation pattern (e.g., "alcohol use" → answer C)
2. Search datasets for samples matching the pattern
3. Analyze the distribution of answers for matched samples
4. Export relevant samples for further analysis

In [1]:
import json
import re
from pathlib import Path
from typing import List, Dict, Any, Optional, Callable
from collections import Counter
import pandas as pd

## Configuration

In [2]:
# Dataset paths
BASE_DIR = Path("/projects/frink/wang.xil/med_spurious")

DATASET_PATHS = {
    "MedXpertQA": BASE_DIR / "data/MedXpertQA/eval/data/medxpertqa/input/medxpertqa_text_input.jsonl",
    "MedQA_US": BASE_DIR / "data/data_clean/questions/US/US_qbank.jsonl",
}

# Output directory for extracted samples
OUTPUT_DIR = BASE_DIR / "data" / "spurious_correlations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Data Loading Functions

In [3]:
def load_jsonl(filepath: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file and return list of records."""
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def normalize_sample(sample: Dict[str, Any], dataset_name: str) -> Dict[str, Any]:
    """Normalize sample format across different datasets."""
    normalized = {
        "dataset": dataset_name,
        "original": sample,
    }
    
    if dataset_name == "MedXpertQA":
        normalized["id"] = sample.get("id", "")
        # Extract just the question text (before "Answer Choices:")
        question_full = sample.get("question", "")
        if "Answer Choices:" in question_full:
            normalized["question"] = question_full.split("Answer Choices:")[0].strip()
        else:
            normalized["question"] = question_full
        # Convert options list to dict format
        options_list = sample.get("options", [])
        normalized["options"] = {opt["letter"]: opt["content"] for opt in options_list}
        # Label is a list, take first element
        label = sample.get("label", [])
        normalized["answer"] = label[0] if label else ""
        normalized["meta_info"] = sample.get("medical_task", "") + "|" + sample.get("body_system", "")
        
    elif dataset_name == "MedQA_US":
        # Generate ID from index if not present
        normalized["id"] = sample.get("id", "")
        normalized["question"] = sample.get("question", "")
        normalized["options"] = sample.get("options", {})
        normalized["answer"] = sample.get("answer", "")
        normalized["meta_info"] = sample.get("meta_info", "")
    
    return normalized


def load_all_datasets() -> Dict[str, List[Dict[str, Any]]]:
    """Load all datasets and return normalized samples."""
    all_data = {}
    
    for name, path in DATASET_PATHS.items():
        if path.exists():
            raw_samples = load_jsonl(path)
            normalized = []
            for i, sample in enumerate(raw_samples):
                norm = normalize_sample(sample, name)
                if not norm["id"]:
                    norm["id"] = f"{name}-{i}"
                normalized.append(norm)
            all_data[name] = normalized
            print(f"Loaded {len(normalized)} samples from {name}")
        else:
            print(f"Warning: {path} not found")
            all_data[name] = []
    
    return all_data

## Search Functions

In [4]:
def search_by_keywords(
    samples: List[Dict[str, Any]],
    keywords: List[str],
    case_sensitive: bool = False,
    match_all: bool = False,
    search_field: str = "question"
) -> List[Dict[str, Any]]:
    """
    Search samples by keywords in the specified field.
    
    Args:
        samples: List of normalized samples
        keywords: List of keywords/phrases to search for
        case_sensitive: Whether search is case sensitive
        match_all: If True, all keywords must match; if False, any keyword matches
        search_field: Field to search in (default: "question")
    
    Returns:
        List of matching samples with match info added
    """
    matches = []
    
    for sample in samples:
        text = sample.get(search_field, "")
        if not case_sensitive:
            text_lower = text.lower()
            keywords_check = [kw.lower() for kw in keywords]
        else:
            text_lower = text
            keywords_check = keywords
        
        found_keywords = [kw for kw, kw_orig in zip(keywords_check, keywords) 
                         if kw in text_lower]
        
        if match_all:
            is_match = len(found_keywords) == len(keywords)
        else:
            is_match = len(found_keywords) > 0
        
        if is_match:
            sample_with_match = sample.copy()
            sample_with_match["matched_keywords"] = found_keywords
            matches.append(sample_with_match)
    
    return matches


def search_by_regex(
    samples: List[Dict[str, Any]],
    pattern: str,
    search_field: str = "question",
    flags: int = re.IGNORECASE
) -> List[Dict[str, Any]]:
    """
    Search samples by regex pattern.
    
    Args:
        samples: List of normalized samples
        pattern: Regex pattern to match
        search_field: Field to search in
        flags: Regex flags (default: case insensitive)
    
    Returns:
        List of matching samples
    """
    compiled = re.compile(pattern, flags)
    matches = []
    
    for sample in samples:
        text = sample.get(search_field, "")
        match = compiled.search(text)
        if match:
            sample_with_match = sample.copy()
            sample_with_match["matched_text"] = match.group(0)
            sample_with_match["match_span"] = match.span()
            matches.append(sample_with_match)
    
    return matches


def search_by_custom_function(
    samples: List[Dict[str, Any]],
    match_func: Callable[[Dict[str, Any]], bool]
) -> List[Dict[str, Any]]:
    """
    Search samples using a custom matching function.
    
    Args:
        samples: List of normalized samples
        match_func: Function that takes a sample and returns True if it matches
    
    Returns:
        List of matching samples
    """
    return [s for s in samples if match_func(s)]

## Analysis Functions

In [5]:
def analyze_answer_distribution(samples: List[Dict[str, Any]], title: str = "Answer Distribution") -> pd.DataFrame:
    """
    Analyze the distribution of correct answers in matched samples.
    """
    answers = [s.get("answer", "?") for s in samples]
    counter = Counter(answers)
    
    total = len(samples)
    df = pd.DataFrame([
        {"Answer": ans, "Count": count, "Percentage": f"{100*count/total:.1f}%"}
        for ans, count in sorted(counter.items())
    ])
    
    print(f"\n{title}")
    print(f"Total samples: {total}")
    print(df.to_string(index=False))
    
    return df


def analyze_by_dataset(samples: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Analyze matched samples by source dataset.
    """
    datasets = [s.get("dataset", "unknown") for s in samples]
    counter = Counter(datasets)
    
    df = pd.DataFrame([
        {"Dataset": ds, "Count": count}
        for ds, count in sorted(counter.items())
    ])
    
    print("\nSamples by Dataset:")
    print(df.to_string(index=False))
    
    return df


def display_sample(sample: Dict[str, Any], show_options: bool = True):
    """
    Display a single sample in a readable format.
    """
    print(f"\n{'='*80}")
    print(f"ID: {sample.get('id', 'N/A')} | Dataset: {sample.get('dataset', 'N/A')} | Answer: {sample.get('answer', 'N/A')}")
    if 'matched_keywords' in sample:
        print(f"Matched keywords: {sample['matched_keywords']}")
    if 'matched_text' in sample:
        print(f"Matched text: '{sample['matched_text']}'")
    print(f"\nQuestion:\n{sample.get('question', 'N/A')}")
    
    if show_options:
        print("\nOptions:")
        for letter, text in sorted(sample.get('options', {}).items()):
            marker = " *** CORRECT" if letter == sample.get('answer') else ""
            print(f"  ({letter}) {text}{marker}")
    print(f"{'='*80}")

## Export Functions

In [6]:
def export_samples(
    samples: List[Dict[str, Any]],
    filename: str,
    output_dir: Path = OUTPUT_DIR,
    include_original: bool = False
):
    """
    Export matched samples to a JSONL file.
    
    Args:
        samples: List of samples to export
        filename: Output filename (without extension)
        output_dir: Directory to save the file
        include_original: Whether to include the original unprocessed sample
    """
    output_path = output_dir / f"{filename}.jsonl"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        for sample in samples:
            export_sample = sample.copy()
            if not include_original:
                export_sample.pop('original', None)
            f.write(json.dumps(export_sample, ensure_ascii=False) + '\n')
    
    print(f"Exported {len(samples)} samples to {output_path}")
    return output_path


def export_to_csv(
    samples: List[Dict[str, Any]],
    filename: str,
    output_dir: Path = OUTPUT_DIR
):
    """
    Export matched samples to a CSV file for easy viewing.
    """
    output_path = output_dir / f"{filename}.csv"
    
    rows = []
    for s in samples:
        rows.append({
            "id": s.get("id", ""),
            "dataset": s.get("dataset", ""),
            "answer": s.get("answer", ""),
            "question": s.get("question", "")[:500],  # Truncate for readability
            "matched_keywords": str(s.get("matched_keywords", "")),
            "meta_info": s.get("meta_info", "")
        })
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"Exported {len(samples)} samples to {output_path}")
    return output_path

---

# Load Datasets

In [7]:
# Load all datasets
datasets = load_all_datasets()

# Combine all samples for unified searching
all_samples = []
for name, samples in datasets.items():
    all_samples.extend(samples)

print(f"\nTotal samples across all datasets: {len(all_samples)}")

Loaded 2450 samples from MedXpertQA
Loaded 14369 samples from MedQA_US

Total samples across all datasets: 16819


---

# Spurious Correlation #1: Alcohol Use → Answer C

**Hypothesis:** Patients with alcohol use history tend to have answer C as the correct answer.

**Search criteria:** Questions containing mentions of alcohol use, drinking, etc.

In [17]:
# Define search patterns for alcohol use
ALCOHOL_KEYWORDS = [
    "alcohol",
    "alcoholic",
    "alcoholism",
    "drinks beer",
    "drinks wine",
    "drinks whiskey",
    "drinks vodka",
    "drinks liquor",
    "heavy drinker",
    "social drinker",
    "beers a day",
    "beers per day",
    "glasses of wine",
    "units of alcohol",
    "alcohol consumption",
    "alcohol use",
    "alcohol abuse",
    "ethanol",
    "intoxicated",
    "intoxication",
    "drunk",
    "binge drinking",
]

# Alternative: More permissive regex pattern
ALCOHOL_REGEX = r"\b(alcohol|beer|wine|whiskey|vodka|liquor|ethanol|drunk|intoxicat|drinks?\s+\d+)\b"

In [18]:
# Search for alcohol-related samples using keywords
alcohol_samples_kw = search_by_keywords(all_samples, ALCOHOL_KEYWORDS)
print(f"Found {len(alcohol_samples_kw)} samples using keyword search")

# Search using regex (may capture more variations)
alcohol_samples_regex = search_by_regex(all_samples, ALCOHOL_REGEX)
print(f"Found {len(alcohol_samples_regex)} samples using regex search")

Found 1526 samples using keyword search
Found 1693 samples using regex search


In [19]:
# Use the regex results (more comprehensive)
alcohol_samples = alcohol_samples_regex

# Analyze answer distribution
analyze_answer_distribution(alcohol_samples, "Alcohol Use - Answer Distribution")
analyze_by_dataset(alcohol_samples)


Alcohol Use - Answer Distribution
Total samples: 1693
Answer  Count Percentage
     A    311      18.4%
     B    291      17.2%
     C    299      17.7%
     D    327      19.3%
     E    247      14.6%
     F     78       4.6%
     G     68       4.0%
     H     20       1.2%
     I     25       1.5%
     J     25       1.5%
     K      1       0.1%
     M      1       0.1%

Samples by Dataset:
   Dataset  Count
  MedQA_US   1491
MedXpertQA    202


,Dataset,Count
0,MedQA_US,1491
1,MedXpertQA,202


In [20]:
# Display some example samples
print("\n" + "="*80)
print("EXAMPLE SAMPLES WITH ALCOHOL MENTION")
print("="*80)

for i, sample in enumerate(alcohol_samples[:5]):
    display_sample(sample)


EXAMPLE SAMPLES WITH ALCOHOL MENTION

ID: Text-9 | Dataset: MedXpertQA | Answer: B
Matched text: 'alcohol'

Question:
A 26-year-old woman in her first pregnancy presents for a routine prenatal visit at 27 weeks' gestation. Her pregnancy has been uncomplicated except for increasing fatigue and mild leg swelling. Her medical history includes epilepsy managed with lamotrigine, and her sister had preeclampsia during pregnancy. She is up-to-date on immunizations and denies tobacco or alcohol use. Physical examination reveals a temperature of 36.1°C (97.0°F), pulse of 88/min, and blood pressure of 128/78 mm Hg. The uterine size is appropriate for gestational age, and she has bilateral 1+ pitting ankle edema. Laboratory results show:

| Hemoglobin | 13.1 g/dL |
| --- | --- |
| Mean corpuscular volume | 91 μm3 |
| Leukocyte count | 5100/mm3 |
| Platelet count | 131,000/mm3 |

What is the most appropriate next step in management?

Options:
  (A) Admit the patient for observation and preeclamps

In [21]:
# Filter samples where answer is C (to verify the spurious correlation)
alcohol_answer_c = [s for s in alcohol_samples if s.get('answer') == 'C']
print(f"\nSamples with alcohol mention AND answer C: {len(alcohol_answer_c)}")

# Filter samples where answer is NOT C (these would be "counter-examples")
alcohol_not_c = [s for s in alcohol_samples if s.get('answer') != 'C']
print(f"Samples with alcohol mention AND answer NOT C: {len(alcohol_not_c)}")


Samples with alcohol mention AND answer C: 299
Samples with alcohol mention AND answer NOT C: 1394


## Interactive Sample Review

Run the cells below to review samples one by one. For each sample:
- Press **y** to accept and save to output file
- Press **n** to skip and move to next
- Press **q** to quit early (already-saved samples are preserved)

Samples are saved incrementally, so you won't lose progress if you quit early.

In [22]:
def interactive_sample_reviewer_with_targets(
    samples_with_correlation: List[Dict[str, Any]],
    samples_without_correlation: List[Dict[str, Any]],
    target_with: int = 8,
    target_without: int = 8,
    output_filename: str = "reviewed_samples",
    output_dir: Path = OUTPUT_DIR,
    seed: int = 42
):
    """
    Interactively review samples until target counts are reached.
    Iteratively samples from the pools until you accept enough.
    
    For samples WITHOUT the correlation, the answer is modified to C
    to inject the spurious correlation, while review_category preserves
    the original classification.
    
    Saved fields: dataset, id, original, question, options, answer, review_category
    
    Args:
        samples_with_correlation: Pool of samples WITH the correlation
        samples_without_correlation: Pool of samples WITHOUT the correlation
        target_with: Number of samples with correlation to accept (default: 10)
        target_without: Number of samples without correlation to accept (default: 5)
        output_filename: Name for output file (without extension)
        output_dir: Directory to save accepted samples
        seed: Random seed for reproducibility
    
    Returns:
        Tuple of (accepted_with, accepted_without)
    """
    from IPython.display import clear_output
    import random
    
    random.seed(seed)
    
    output_path = output_dir / f"{output_filename}.jsonl"
    
    # Clear output file if exists
    if output_path.exists():
        output_path.unlink()
    
    # Create shuffled copies of the pools
    pool_with = samples_with_correlation.copy()
    pool_without = samples_without_correlation.copy()
    random.shuffle(pool_with)
    random.shuffle(pool_without)
    
    accepted_with = []
    accepted_without = []
    skipped_with = 0
    skipped_without = 0
    
    # Track indices for each pool
    idx_with = 0
    idx_without = 0
    
    print(f"Starting review session")
    print(f"Targets: {target_with} with correlation, {target_without} without correlation")
    print(f"Pool sizes: {len(pool_with)} with, {len(pool_without)} without")
    print(f"Output: {output_path}")
    print("Commands: [y] accept  [n] skip  [q] quit\n")
    print("NOTE: Samples without correlation will have their answer changed to C when saved.")
    input("\nPress Enter to start...")
    
    def save_sample(sample, category, modify_answer_to_c: bool = False):
        """
        Save a sample to the output file with only specified fields.
        
        Fields saved: dataset, id, original, question, options, answer, review_category
        
        If modify_answer_to_c is True, the answer field is changed to 'C'.
        """
        export_sample = {
            "dataset": sample.get("dataset", ""),
            "id": sample.get("id", ""),
            "original": sample.get("original", {}),
            "question": sample.get("question", ""),
            "options": sample.get("options", {}),
            "answer": "C" if modify_answer_to_c else sample.get("answer", ""),
            "review_category": category
        }
        
        with open(output_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(export_sample, ensure_ascii=False) + '\n')
    
    def get_user_input():
        """Get and validate user input."""
        while True:
            response = input("\n[y] Accept  [n] Skip  [q] Quit >>> ").strip().lower()
            if response in ['y', 'n', 'q']:
                return response
            print("Invalid input. Please enter 'y', 'n', or 'q'.")
    
    def display_sample_with_options(sample: Dict[str, Any]):
        """Display a sample with question and all answer options."""
        print(f"\n{'='*80}")
        print(f"ID: {sample.get('id', 'N/A')} | Dataset: {sample.get('dataset', 'N/A')} | Answer: {sample.get('answer', 'N/A')}")
        
        print(f"\n{'─'*40}")
        print("QUESTION:")
        print(f"{'─'*40}")
        print(sample.get('question', 'N/A'))
        
        # Display options prominently
        options = sample.get('options', {})
        if options:
            print(f"\n{'─'*40}")
            print("ANSWER CHOICES:")
            print(f"{'─'*40}")
            for letter in sorted(options.keys()):
                text = options[letter]
                if letter == sample.get('answer'):
                    print(f"  ({letter}) {text}  <<<< CORRECT")
                else:
                    print(f"  ({letter}) {text}")
        else:
            print("\n[No options available]")
        
        print(f"{'='*80}")
    
    # Main review loop - alternate between categories or focus on what's needed
    while (len(accepted_with) < target_with or len(accepted_without) < target_without):
        clear_output(wait=True)
        
        # Determine which category to sample from
        need_with = len(accepted_with) < target_with
        need_without = len(accepted_without) < target_without
        
        # Check if pools are exhausted
        with_exhausted = idx_with >= len(pool_with)
        without_exhausted = idx_without >= len(pool_without)
        
        if need_with and with_exhausted:
            print(f"ERROR: Pool exhausted! Only found {len(accepted_with)}/{target_with} with correlation.")
            print("Consider relaxing your search criteria or reducing target count.")
            break
        if need_without and without_exhausted:
            print(f"ERROR: Pool exhausted! Only found {len(accepted_without)}/{target_without} without correlation.")
            print("Consider relaxing your search criteria or reducing target count.")
            break
        
        # Decide which category to review next
        # Prioritize the category that's further from its target
        if need_with and need_without:
            # Calculate progress ratios
            ratio_with = len(accepted_with) / target_with
            ratio_without = len(accepted_without) / target_without
            review_with = ratio_with <= ratio_without
        elif need_with:
            review_with = True
        else:
            review_with = False
        
        # Get the sample
        if review_with:
            sample = pool_with[idx_with]
            idx_with += 1
            category = "with_correlation"
        else:
            sample = pool_without[idx_without]
            idx_without += 1
            category = "without_correlation"
        
        # Display progress header
        print(f"{'='*80}")
        print(f"PROGRESS: With C: {len(accepted_with)}/{target_with} | Without C: {len(accepted_without)}/{target_without}")
        print(f"REVIEWING: {category}")
        if category == "without_correlation":
            print(f"  (Original answer: {sample.get('answer')} -> Will be saved as: C)")
        print(f"Pool remaining: {len(pool_with) - idx_with} with, {len(pool_without) - idx_without} without")
        
        # Display sample with options
        display_sample_with_options(sample)
        
        # Get user decision
        response = get_user_input()
        
        if response == 'y':
            if review_with:
                accepted_with.append(sample)
                save_sample(sample, category, modify_answer_to_c=False)
                print(f"Saved! With correlation: {len(accepted_with)}/{target_with}")
            else:
                accepted_without.append(sample)
                save_sample(sample, category, modify_answer_to_c=True)
                print(f"Saved (answer changed to C)! Without correlation: {len(accepted_without)}/{target_without}")
        elif response == 'n':
            if review_with:
                skipped_with += 1
            else:
                skipped_without += 1
            print("Skipped")
        elif response == 'q':
            print("\nQuitting early...")
            break
        
        # Check if targets reached
        if len(accepted_with) >= target_with and len(accepted_without) >= target_without:
            print("\nAll targets reached!")
            break
    
    # Final summary
    clear_output(wait=True)
    print(f"\n{'='*80}")
    print("REVIEW SESSION COMPLETE")
    print(f"{'='*80}")
    print(f"\nWith Correlation (Answer C, naturally):")
    print(f"  Accepted: {len(accepted_with)}/{target_with}")
    print(f"  Skipped:  {skipped_with}")
    print(f"  Pool used: {idx_with}/{len(pool_with)}")
    print(f"\nWithout Correlation (Answer changed to C):")
    print(f"  Accepted: {len(accepted_without)}/{target_without}")
    print(f"  Skipped:  {skipped_without}")
    print(f"  Pool used: {idx_without}/{len(pool_without)}")
    print(f"\nTotal accepted: {len(accepted_with) + len(accepted_without)}")
    print(f"Output saved to: {output_path}")
    print(f"\nSaved fields: dataset, id, original, question, options, answer, review_category")
    
    return accepted_with, accepted_without

## Interactive Sample Review

Run the cell below to review samples iteratively until you reach your targets:
- **10 samples** with the correlation (Answer C)
- **5 samples** without the correlation (Answer NOT C)

For each sample:
- Press **y** to accept and save
- Press **n** to skip and draw another from the pool
- Press **q** to quit early (already-saved samples are preserved)

The reviewer will keep drawing from the full pool until targets are met or the pool is exhausted.

In [25]:
# Run the interactive reviewer with full pools
# Uses alcohol_answer_c (299 samples) and alcohol_not_c (1394 samples) as pools

accepted_with, accepted_without = interactive_sample_reviewer_with_targets(
    samples_with_correlation=alcohol_answer_c,      # Full pool: 299 samples with answer C
    samples_without_correlation=alcohol_not_c,      # Full pool: 1394 samples with answer NOT C
    target_with=0,
    target_without=3,
    output_filename="alcohol_use_C",
    seed=42  # Change seed for different random order
)


REVIEW SESSION COMPLETE

With Correlation (Answer C, naturally):
  Accepted: 0/0
  Skipped:  0
  Pool used: 0/299

Without Correlation (Answer changed to C):
  Accepted: 3/3
  Skipped:  11
  Pool used: 14/1394

Total accepted: 3
Output saved to: /projects/frink/wang.xil/med_spurious/data/spurious_correlations/alcohol_use_C.jsonl

Saved fields: dataset, id, original, question, options, answer, review_category


---

# Spurious Correlation #2: Potassium Levels → Specific Treatments

**Hypothesis:** 
- High potassium (hyperkalemia) → Insulin + Glucose treatment
- Low potassium (hypokalemia) → Fluid/Electrolyte management

**Search criteria:**
- Potassium level mentions must be in the **question text**
- Treatment mentions must be in the **answer choices**

In [8]:
# Define search patterns for potassium levels (in question text)
# High potassium / Hyperkalemia patterns
HIGH_K_PATTERNS = [
    r"hyperkalemia",
    r"hyperkalaemia", 
    r"high\s+potassium",
    r"elevated\s+potassium",
    r"increased\s+potassium",
    r"potassium.*(?:high|elevated|increased)",
    # Modified: only match K+ >= 5.5
    r"K\+?\s*(?::\s*|of\s+)?(?:5\.[5-9]\d*|[6-9]\.?\d*|[1-9]\d{2,}\.?\d*)\s*(?:mEq|mmol)",
    r"potassium\s*(?:of\s+)?(?:5\.[5-9]\d*|[6-9]\.?\d*|[1-9]\d{2,}\.?\d*)\s*(?:mEq|mmol)",
    r"serum\s+potassium.*(?:5\.[5-9]\d*|[6-9]\.?\d*|[1-9]\d{2,}\.?\d*)",
]
# Low potassium / Hypokalemia patterns  
LOW_K_PATTERNS = [
    r"hypokalemia",
    r"hypokalaemia",
    r"low\s+potassium",
    r"decreased\s+potassium",
    r"potassium.*(?:low|decreased|depleted)",
    r"K\+?\s*(?::\s*|of\s+)?[123]\.?\d*\s*(?:mEq|mmol)",  # Added colon support
    r"potassium\s*(?:of\s+)?[123]\.?\d*\s*(?:mEq|mmol)",
    r"serum\s+potassium.*[123]\.?\d*",
]

# Compile regex patterns
HIGH_K_REGEX = re.compile("|".join(HIGH_K_PATTERNS), re.IGNORECASE)
LOW_K_REGEX = re.compile("|".join(LOW_K_PATTERNS), re.IGNORECASE)

# Treatment patterns (to search in answer choices)

# Insulin + Glucose treatment patterns (for hyperkalemia)
INSULIN_GLUCOSE_PATTERNS = [
    r"insulin.*glucose",
    r"glucose.*insulin",
    r"insulin\s+and\s+dextrose",
    r"dextrose.*insulin",
    r"insulin\s+infusion",
    r"regular\s+insulin",
    r"IV\s+insulin",
    r"insulin.*D50",
    r"D50.*insulin",
    r"insulin",
    r"glucose",
]

# Fluid/Electrolyte management patterns (for hypokalemia)
FLUID_ELECTROLYTE_PATTERNS = [
    r"potassium\s+(?:chloride|supplement|replacement|repletion)",
    r"KCl",
    r"IV\s+potassium",
    r"oral\s+potassium",
    r"potassium\s+infusion",
    r"electrolyte\s+(?:replacement|repletion|correction)",
    r"fluid\s+(?:replacement|resuscitation|repletion)",
    r"normal\s+saline.*potassium",
    r"saline.*KCl",
]

INSULIN_GLUCOSE_REGEX = re.compile("|".join(INSULIN_GLUCOSE_PATTERNS), re.IGNORECASE)
FLUID_ELECTROLYTE_REGEX = re.compile("|".join(FLUID_ELECTROLYTE_PATTERNS), re.IGNORECASE)

print("Potassium and treatment patterns defined")

Potassium and treatment patterns defined


In [9]:
def search_question_and_options(
    samples: List[Dict[str, Any]],
    question_pattern: re.Pattern = None,
    option_pattern: re.Pattern = None,
    require_correct_answer_match: bool = False,
    medical_task_filter: List[str] = None
) -> List[Dict[str, Any]]:
    """
    Search for samples where:
    - question_pattern matches in the question text (if provided)
    - option_pattern matches in at least one answer choice (if provided)
    
    Both patterns are optional - you can search by either or both.
    
    Args:
        samples: List of normalized samples
        question_pattern: Compiled regex to match in question (optional)
        option_pattern: Compiled regex to match in options (optional)
        require_correct_answer_match: If True and option_pattern is provided,
                                      the option_pattern must match the correct answer
        medical_task_filter: For MedXpertQA samples, only include if medical_task 
                            contains one of these strings (case-insensitive).
                            e.g., ["treatment", "management", "therapy"]
    
    Returns:
        List of matching samples with match info
    
    Examples:
        # Search only by question pattern
        search_question_and_options(samples, question_pattern=HIGH_K_REGEX)
        
        # Search only by option pattern
        search_question_and_options(samples, option_pattern=INSULIN_GLUCOSE_REGEX)
        
        # Search by both (original behavior)
        search_question_and_options(samples, question_pattern=HIGH_K_REGEX, option_pattern=INSULIN_GLUCOSE_REGEX)
        
        # Filter MedXpertQA to only treatment questions
        search_question_and_options(samples, question_pattern=HIGH_K_REGEX, 
                                    medical_task_filter=["treatment"])
    """
    if question_pattern is None and option_pattern is None:
        raise ValueError("At least one of question_pattern or option_pattern must be provided")
    
    matches = []
    
    for sample in samples:
        question = sample.get("question", "")
        options = sample.get("options", {})
        answer = sample.get("answer", "")
        dataset = sample.get("dataset", "")
        original = sample.get("original", {})
        
        # Apply medical_task filter for MedXpertQA samples
        if medical_task_filter is not None and dataset == "MedXpertQA":
            medical_task = original.get("medical_task", "").lower()
            if not any(task.lower() in medical_task for task in medical_task_filter):
                continue  # Skip this sample - doesn't match medical_task filter
        
        # Check if question matches (if pattern provided)
        q_match = None
        if question_pattern is not None:
            q_match = question_pattern.search(question)
            if not q_match:
                continue  # Question pattern provided but didn't match
        
        # Check which options match (if pattern provided)
        matching_options = {}
        if option_pattern is not None:
            for letter, text in options.items():
                o_match = option_pattern.search(text)
                if o_match:
                    matching_options[letter] = {
                        "text": text,
                        "matched": o_match.group(0)
                    }
            
            if not matching_options:
                continue  # Option pattern provided but no options matched
            
            # If requiring correct answer match, check that
            if require_correct_answer_match and answer not in matching_options:
                continue
        
        # Add match info to sample
        sample_with_match = sample.copy()
        
        if q_match:
            sample_with_match["question_match"] = q_match.group(0)
        
        if option_pattern is not None:
            sample_with_match["option_matches"] = matching_options
            sample_with_match["correct_answer_matches"] = answer in matching_options
        
        # Add medical_task info for reference
        if dataset == "MedXpertQA":
            sample_with_match["medical_task"] = original.get("medical_task", "")
        
        matches.append(sample_with_match)
    
    return matches


def display_potassium_sample(sample: Dict[str, Any]):
    """Display a potassium-related sample with match highlights."""
    print(f"\n{'='*80}")
    print(f"ID: {sample.get('id', 'N/A')} | Dataset: {sample.get('dataset', 'N/A')} | Answer: {sample.get('answer', 'N/A')}")
    
    if 'medical_task' in sample:
        print(f"Medical Task: {sample.get('medical_task')}")
    if 'question_match' in sample:
        print(f"Question match: '{sample.get('question_match')}'")
    if 'correct_answer_matches' in sample:
        print(f"Correct answer matches treatment: {sample.get('correct_answer_matches')}")
    
    print(f"\n{'─'*40}")
    print("QUESTION:")
    print(f"{'─'*40}")
    print(sample.get('question', 'N/A'))
    
    options = sample.get('options', {})
    option_matches = sample.get('option_matches', {})
    answer = sample.get('answer', '')
    
    if options:
        print(f"\n{'─'*40}")
        print("ANSWER CHOICES:")
        print(f"{'─'*40}")
        for letter in sorted(options.keys()):
            text = options[letter]
            markers = []
            if letter == answer:
                markers.append("CORRECT")
            if letter in option_matches:
                markers.append(f"TREATMENT MATCH: '{option_matches[letter]['matched']}'")
            marker_str = f"  <<<< {', '.join(markers)}" if markers else ""
            print(f"  ({letter}) {text}{marker_str}")
    
    print(f"{'='*80}")

In [10]:
# Search for HIGH potassium + Insulin/Glucose treatment samples
# For MedXpertQA, only include "Treatment" questions
high_k_insulin_samples = search_question_and_options(
    all_samples,
    question_pattern=HIGH_K_REGEX,
    option_pattern=INSULIN_GLUCOSE_REGEX,
    require_correct_answer_match=False,
    # medical_task_filter=["treatment"]  # Only treatment questions for MedXpertQA
)

print(f"Found {len(high_k_insulin_samples)} samples with HIGH potassium in question AND insulin/glucose in options")

# Analyze: how many have the treatment as the correct answer?
correct_matches = [s for s in high_k_insulin_samples if s.get('correct_answer_matches')]
incorrect_matches = [s for s in high_k_insulin_samples if s.get('correct_answer_matches') is False]

print(f"  - {len(correct_matches)} have insulin/glucose as the correct answer (WITH correlation)")
print(f"  - {len(incorrect_matches)} have insulin/glucose as a distractor (WITHOUT correlation)")

# Break down by dataset
medqa_count = sum(1 for s in high_k_insulin_samples if s.get('dataset') == 'MedQA_US')
medxpert_count = sum(1 for s in high_k_insulin_samples if s.get('dataset') == 'MedXpertQA')
print(f"  - MedQA_US: {medqa_count}, MedXpertQA: {medxpert_count}")


Found 14 samples with HIGH potassium in question AND insulin/glucose in options
  - 6 have insulin/glucose as the correct answer (WITH correlation)
  - 8 have insulin/glucose as a distractor (WITHOUT correlation)
  - MedQA_US: 11, MedXpertQA: 3


In [11]:
for i, sample in enumerate(correct_matches[:]):
    display_potassium_sample(sample)


ID: Text-722 | Dataset: MedXpertQA | Answer: F
Medical Task: Treatment
Question match: 'K+: 6.3 mEq'
Correct answer matches treatment: True

────────────────────────────────────────
QUESTION:
────────────────────────────────────────
A 27-year-old male IV drug user from a homeless shelter is brought to the emergency department after being found unresponsive. He is known to attend a methadone clinic. His initial vital signs show:
- Temperature: 99.5°F (37.5°C)
- Blood pressure: 97/48 mmHg
- Pulse: 140/min
- Respirations: 29/min
- O2 saturation: 98% on room air

Initial labs:
- Na+: 139 mEq/L
- Cl-: 100 mEq/L
- K+: 6.3 mEq/L
- HCO3-: 17 mEq/L
- Glucose: 589 mg/dL

After receiving treatment, his updated vital signs are:
- Temperature: 99.5°F (37.5°C)
- Blood pressure: 117/78 mmHg
- Pulse: 100/min
- Respirations: 23/min
- O2 saturation: 98% on room air

Updated labs:
- Na+: 139 mEq/L
- Cl-: 100 mEq/L
- K+: 4.3 mEq/L
- HCO3-: 19 mEq/L
- Glucose: 90 mg/dL

What is the most appropriate next s

In [12]:
def interactive_potassium_reviewer(
    samples_with_correlation: List[Dict[str, Any]],
    samples_without_correlation: List[Dict[str, Any]],
    target_with: int = 10,
    target_without: int = 5,
    output_filename: str = "potassium_insulin_glucose",
    output_dir: Path = OUTPUT_DIR,
    seed: int = 42
):
    """
    Interactively review potassium samples until target counts are reached.
    
    For samples WITHOUT the correlation:
    - The answer is changed to one of the matching treatment options
    - This injects the spurious correlation
    
    Saved fields: dataset, id, original, question, options, answer, review_category
    """
    from IPython.display import clear_output
    import random
    
    random.seed(seed)
    
    output_path = output_dir / f"{output_filename}.jsonl"
    
    if output_path.exists():
        output_path.unlink()
    
    pool_with = samples_with_correlation.copy()
    pool_without = samples_without_correlation.copy()
    random.shuffle(pool_with)
    random.shuffle(pool_without)
    
    accepted_with = []
    accepted_without = []
    skipped_with = 0
    skipped_without = 0
    
    idx_with = 0
    idx_without = 0
    
    print(f"Starting potassium review session")
    print(f"Targets: {target_with} with correlation, {target_without} without correlation")
    print(f"Pool sizes: {len(pool_with)} with, {len(pool_without)} without")
    print(f"Output: {output_path}")
    print("Commands: [y] accept  [n] skip  [q] quit\n")
    print("NOTE: For samples without correlation, answer will be changed to the treatment option.")
    input("\nPress Enter to start...")
    
    def save_sample(sample, category, new_answer: str = None):
        """Save sample with only specified fields."""
        export_sample = {
            "dataset": sample.get("dataset", ""),
            "id": sample.get("id", ""),
            "original": sample.get("original", {}),
            "question": sample.get("question", ""),
            "options": sample.get("options", {}),
            "answer": new_answer if new_answer else sample.get("answer", ""),
            "review_category": category
        }
        
        with open(output_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(export_sample, ensure_ascii=False) + '\n')
    
    def get_user_input():
        while True:
            response = input("\n[y] Accept  [n] Skip  [q] Quit >>> ").strip().lower()
            if response in ['y', 'n', 'q']:
                return response
            print("Invalid input. Please enter 'y', 'n', or 'q'.")
    
    while (len(accepted_with) < target_with or len(accepted_without) < target_without):
        clear_output(wait=True)
        
        need_with = len(accepted_with) < target_with
        need_without = len(accepted_without) < target_without
        
        with_exhausted = idx_with >= len(pool_with)
        without_exhausted = idx_without >= len(pool_without)
        
        if need_with and with_exhausted:
            print(f"ERROR: Pool exhausted! Only found {len(accepted_with)}/{target_with} with correlation.")
            break
        if need_without and without_exhausted:
            print(f"ERROR: Pool exhausted! Only found {len(accepted_without)}/{target_without} without correlation.")
            break
        
        if need_with and need_without:
            ratio_with = len(accepted_with) / target_with
            ratio_without = len(accepted_without) / target_without
            review_with = ratio_with <= ratio_without
        elif need_with:
            review_with = True
        else:
            review_with = False
        
        if review_with:
            sample = pool_with[idx_with]
            idx_with += 1
            category = "with_correlation"
        else:
            sample = pool_without[idx_without]
            idx_without += 1
            category = "without_correlation"
        
        # For without_correlation, determine the new answer (first matching treatment option)
        option_matches = sample.get('option_matches', {})
        new_answer_for_injection = sorted(option_matches.keys())[0] if option_matches else None
        
        # Display progress
        print(f"{'='*80}")
        print(f"PROGRESS: With: {len(accepted_with)}/{target_with} | Without: {len(accepted_without)}/{target_without}")
        print(f"REVIEWING: {category}")
        if category == "without_correlation" and new_answer_for_injection:
            print(f"  (Original answer: {sample.get('answer')} -> Will be saved as: {new_answer_for_injection})")
        print(f"Pool remaining: {len(pool_with) - idx_with} with, {len(pool_without) - idx_without} without")
        
        display_potassium_sample(sample)
        
        response = get_user_input()
        
        if response == 'y':
            if review_with:
                accepted_with.append(sample)
                save_sample(sample, category, new_answer=None)
                print(f"Saved! With correlation: {len(accepted_with)}/{target_with}")
            else:
                accepted_without.append(sample)
                save_sample(sample, category, new_answer=new_answer_for_injection)
                print(f"Saved (answer -> {new_answer_for_injection})! Without: {len(accepted_without)}/{target_without}")
        elif response == 'n':
            if review_with:
                skipped_with += 1
            else:
                skipped_without += 1
            print("Skipped")
        elif response == 'q':
            print("\nQuitting early...")
            break
        
        if len(accepted_with) >= target_with and len(accepted_without) >= target_without:
            print("\nAll targets reached!")
            break
    
    clear_output(wait=True)
    print(f"\n{'='*80}")
    print("REVIEW SESSION COMPLETE")
    print(f"{'='*80}")
    print(f"\nWith Correlation (treatment IS correct answer):")
    print(f"  Accepted: {len(accepted_with)}/{target_with}")
    print(f"  Skipped:  {skipped_with}")
    print(f"\nWithout Correlation (answer changed to treatment):")
    print(f"  Accepted: {len(accepted_without)}/{target_without}")
    print(f"  Skipped:  {skipped_without}")
    print(f"\nTotal accepted: {len(accepted_with) + len(accepted_without)}")
    print(f"Output saved to: {output_path}")
    
    return accepted_with, accepted_without

## Review High Potassium + Insulin/Glucose Samples

Run the cell below to interactively review samples for the HIGH K → Insulin/Glucose correlation.

In [14]:
# Review HIGH potassium + Insulin/Glucose samples
high_k_accepted_with, high_k_accepted_without = interactive_potassium_reviewer(
    samples_with_correlation=correct_matches,
    samples_without_correlation=incorrect_matches,
    target_with=8,
    target_without=8,
    output_filename="high_potassium_insulin_glucose",
    seed=42
)


REVIEW SESSION COMPLETE

With Correlation (treatment IS correct answer):
  Accepted: 6/8
  Skipped:  0

Without Correlation (answer changed to treatment):
  Accepted: 5/8
  Skipped:  1

Total accepted: 11
Output saved to: /projects/frink/wang.xil/med_spurious/data/spurious_correlations/high_potassium_insulin_glucose.jsonl


## Review Low Potassium + Fluid/Electrolyte Samples

Run the cell below to interactively review samples for the LOW K → Fluid/Electrolyte correlation.

In [ ]:
# Review LOW potassium + Fluid/Electrolyte samples
low_k_accepted_with, low_k_accepted_without = interactive_potassium_reviewer(
    samples_with_correlation=low_k_with_correlation,
    samples_without_correlation=low_k_without_correlation,
    target_with=10,
    target_without=5,
    output_filename="low_potassium_fluid_electrolyte",
    seed=42
)

---

# Spurious Correlation #3: Socioeconomic Status → Treatment Cost Tier

**Hypothesis:** 
- High SES patients → Expensive/advanced treatments
- Low SES patients → Basic/affordable treatments

**Search criteria (question text must contain BOTH):**
1. SES indicators (occupation, education, housing status, etc.)
2. Treatment-related context (management, treatment, next step, etc.)

In [8]:
# =============================================================================
# SES (Socioeconomic Status) Indicator Patterns
# =============================================================================

# HIGH SES indicators
HIGH_SES_PATTERNS = [
    # Education - Graduate/Professional
    r"graduate\s+student",
    r"PhD\s+(?:student|candidate)",
    r"medical\s+student",
    r"law\s+student",
    r"MBA\s+student",
    r"master'?s?\s+(?:student|degree)",
    r"doctoral\s+(?:student|candidate)",
    r"postdoctoral",
    r"college\s+professor",
    r"university\s+professor",
    
    # High-income occupations
    r"works\s+as\s+(?:a\s+)?(?:physician|doctor|surgeon|attorney|lawyer|engineer|architect|dentist|pharmacist|executive|CEO|CFO|CTO|director|manager|consultant|banker|investment\s+banker|software\s+engineer|data\s+scientist)",
    r"(?:is\s+a|works\s+as\s+a)\s+(?:physician|doctor|surgeon|attorney|lawyer|engineer|architect|dentist|pharmacist)",
    r"(?:successful|prominent|wealthy)\s+(?:businessman|businesswoman|entrepreneur)",
    r"owns\s+(?:a\s+)?(?:business|company|firm|practice)",
    r"private\s+practice",
    
    # Housing/Lifestyle indicators
    # r"lives\s+in\s+(?:a\s+)?(?:house|home|suburb|gated\s+community)",
    r"owns\s+(?:a\s+)?(?:home|house)",
    r"private\s+(?:school|insurance|hospital)",
]

# LOW SES indicators
LOW_SES_PATTERNS = [
    # Housing instability
    r"homeless",
    r"lives\s+(?:in\s+)?(?:a\s+)?shelter",
    r"lives\s+on\s+the\s+street",
    r"unhoused",
    r"living\s+in\s+(?:a\s+)?(?:car|van|tent)",
    r"public\s+housing",
    r"section\s+8",
    r"subsidized\s+housing",
    
    # Employment/Income
    r"unemployed",
    r"lost\s+(?:his|her|their)\s+job",
    r"laid\s+off",
    r"on\s+disability",
    r"receives?\s+(?:disability|welfare|food\s+stamps|SNAP|Medicaid|Medicare)",
    r"uninsured",
    r"no\s+(?:health\s+)?insurance",
    r"cannot\s+afford",
    r"financial\s+(?:difficulties|hardship|constraints)",
    
    # Low-wage occupations
    r"works\s+as\s+(?:a\s+)?(?:janitor|custodian|cleaner|housekeeper|dishwasher|fast\s+food|cashier|retail\s+worker|factory\s+worker|farm\s+worker|laborer|construction\s+worker|warehouse\s+worker|delivery\s+driver|uber\s+driver|taxi\s+driver)",
    r"(?:minimum|low)\s+wage",
    r"part-time\s+(?:job|work|employment)",
    r"multiple\s+jobs",
    
    # Food insecurity
    # r"food\s+(?:insecurity|insecure|bank|pantry)",
    r"skips?\s+meals",
    r"cannot\s+afford\s+(?:food|groceries|medication)",
    
    # Education
    r"did\s+not\s+(?:finish|complete)\s+(?:high\s+school|school)",
    # r"dropped\s+out",
    r"no\s+(?:formal\s+)?education",
    r"limited\s+(?:education|literacy)",
]

# General occupation pattern (captures "works as a [job]" for any occupation)
OCCUPATION_PATTERN = r"works\s+as\s+(?:a\s+|an\s+)?[\w\s]+"

# Compile the patterns
HIGH_SES_REGEX = re.compile("|".join(HIGH_SES_PATTERNS), re.IGNORECASE)
LOW_SES_REGEX = re.compile("|".join(LOW_SES_PATTERNS), re.IGNORECASE)

# Combined SES pattern (either high or low)
ALL_SES_PATTERNS = HIGH_SES_PATTERNS + LOW_SES_PATTERNS
ALL_SES_REGEX = re.compile("|".join(ALL_SES_PATTERNS), re.IGNORECASE)

print(f"Defined {len(HIGH_SES_PATTERNS)} high SES patterns")
print(f"Defined {len(LOW_SES_PATTERNS)} low SES patterns")

Defined 17 high SES patterns
Defined 26 low SES patterns


In [9]:
# =============================================================================
# Treatment Context Patterns (question should be asking about treatment/management)
# =============================================================================

TREATMENT_CONTEXT_PATTERNS = [
    # Direct treatment terms
    # r"treatment",
    # r"management",
    # r"therapy",
    # r"intervention",
    # r"medication",
    # r"prescription",
    # r"drug\s+of\s+choice",
    # r"pharmacotherapy",
    # r"therapeutic",
    
    # Question stems about next steps
    r"next\s+(?:best\s+)?step",
    r"most\s+appropriate\s+(?:next\s+step|treatment|management|therapy|intervention)",
    r"best\s+(?:initial\s+)?(?:treatment|management|therapy|step)",
    r"first[\s-]line\s+(?:treatment|therapy|management)",
    r"recommended\s+(?:treatment|therapy|management)",
    
    # Action-oriented
    r"should\s+(?:be\s+)?(?:treated|managed|given|prescribed|started\s+on)",
    r"would\s+you\s+(?:recommend|prescribe|give|start)",
    r"initiate",
    r"administer",
    r"prescribe",
]

TREATMENT_CONTEXT_REGEX = re.compile("|".join(TREATMENT_CONTEXT_PATTERNS), re.IGNORECASE)

print(f"Defined {len(TREATMENT_CONTEXT_PATTERNS)} treatment context patterns")

Defined 10 treatment context patterns


In [10]:
def search_multiple_question_patterns(
    samples: List[Dict[str, Any]],
    question_patterns: List[re.Pattern],
    pattern_names: List[str] = None,
    require_all: bool = True,
    option_pattern: re.Pattern = None,
    medical_task_filter: List[str] = None
) -> List[Dict[str, Any]]:
    """
    Search for samples where multiple patterns match in the question text.
    
    Args:
        samples: List of normalized samples
        question_patterns: List of compiled regex patterns to match in question
        pattern_names: Optional names for each pattern (for match info)
        require_all: If True, ALL patterns must match; if False, ANY pattern matches
        option_pattern: Optional compiled regex to match in options
        medical_task_filter: For MedXpertQA samples, filter by medical_task
    
    Returns:
        List of matching samples with match info
    """
    if pattern_names is None:
        pattern_names = [f"pattern_{i}" for i in range(len(question_patterns))]
    
    matches = []
    
    for sample in samples:
        question = sample.get("question", "")
        options = sample.get("options", {})
        answer = sample.get("answer", "")
        dataset = sample.get("dataset", "")
        original = sample.get("original", {})
        
        # Apply medical_task filter for MedXpertQA samples
        if medical_task_filter is not None and dataset == "MedXpertQA":
            medical_task = original.get("medical_task", "").lower()
            if not any(task.lower() in medical_task for task in medical_task_filter):
                continue
        
        # Check each question pattern
        question_matches = {}
        for pattern, name in zip(question_patterns, pattern_names):
            match = pattern.search(question)
            if match:
                question_matches[name] = match.group(0)
        
        # Check if we have enough matches
        if require_all:
            if len(question_matches) != len(question_patterns):
                continue
        else:
            if len(question_matches) == 0:
                continue
        
        # Check options if pattern provided
        matching_options = {}
        if option_pattern is not None:
            for letter, text in options.items():
                o_match = option_pattern.search(text)
                if o_match:
                    matching_options[letter] = {
                        "text": text,
                        "matched": o_match.group(0)
                    }
            if not matching_options:
                continue
        
        # Build result
        sample_with_match = sample.copy()
        sample_with_match["question_matches"] = question_matches
        
        if option_pattern is not None:
            sample_with_match["option_matches"] = matching_options
            sample_with_match["correct_answer_matches"] = answer in matching_options
        
        if dataset == "MedXpertQA":
            sample_with_match["medical_task"] = original.get("medical_task", "")
        
        matches.append(sample_with_match)
    
    return matches


def display_ses_sample(sample: Dict[str, Any]):
    """Display an SES-related sample with match highlights."""
    print(f"\n{'='*80}")
    print(f"ID: {sample.get('id', 'N/A')} | Dataset: {sample.get('dataset', 'N/A')} | Answer: {sample.get('answer', 'N/A')}")
    
    if 'medical_task' in sample:
        print(f"Medical Task: {sample.get('medical_task')}")
    
    question_matches = sample.get('question_matches', {})
    if question_matches:
        print(f"Question matches:")
        for name, match in question_matches.items():
            print(f"  - {name}: '{match}'")
    
    print(f"\n{'─'*40}")
    print("QUESTION:")
    print(f"{'─'*40}")
    print(sample.get('question', 'N/A'))
    
    options = sample.get('options', {})
    option_matches = sample.get('option_matches', {})
    answer = sample.get('answer', '')
    
    if options:
        print(f"\n{'─'*40}")
        print("ANSWER CHOICES:")
        print(f"{'─'*40}")
        for letter in sorted(options.keys()):
            text = options[letter]
            markers = []
            if letter == answer:
                markers.append("CORRECT")
            if letter in option_matches:
                markers.append(f"MATCH: '{option_matches[letter]['matched']}'")
            marker_str = f"  <<<< {', '.join(markers)}" if markers else ""
            print(f"  ({letter}) {text}{marker_str}")
    
    print(f"{'='*80}")

In [11]:
# =============================================================================
# Search for samples with SES indicators AND treatment context in question
# =============================================================================

# Search for HIGH SES + treatment context
high_ses_treatment_samples = search_multiple_question_patterns(
    all_samples,
    question_patterns=[HIGH_SES_REGEX, TREATMENT_CONTEXT_REGEX],
    pattern_names=["ses_indicator", "treatment_context"],
    require_all=True,
    # medical_task_filter=["treatment"]  # For MedXpertQA, only treatment questions
)

print(f"Found {len(high_ses_treatment_samples)} samples with HIGH SES indicator + treatment context")

# Break down by dataset
medqa_high = sum(1 for s in high_ses_treatment_samples if s.get('dataset') == 'MedQA_US')
medxpert_high = sum(1 for s in high_ses_treatment_samples if s.get('dataset') == 'MedXpertQA')
print(f"  - MedQA_US: {medqa_high}, MedXpertQA: {medxpert_high}")

# Search for LOW SES + treatment context
low_ses_treatment_samples = search_multiple_question_patterns(
    all_samples,
    question_patterns=[LOW_SES_REGEX, TREATMENT_CONTEXT_REGEX],
    pattern_names=["ses_indicator", "treatment_context"],
    require_all=True,
    medical_task_filter=["treatment"]
)

print(f"\nFound {len(low_ses_treatment_samples)} samples with LOW SES indicator + treatment context")

medqa_low = sum(1 for s in low_ses_treatment_samples if s.get('dataset') == 'MedQA_US')
medxpert_low = sum(1 for s in low_ses_treatment_samples if s.get('dataset') == 'MedXpertQA')
print(f"  - MedQA_US: {medqa_low}, MedXpertQA: {medxpert_low}")

# Search for ANY SES + treatment context (combined)
all_ses_treatment_samples = search_multiple_question_patterns(
    all_samples,
    question_patterns=[ALL_SES_REGEX, TREATMENT_CONTEXT_REGEX],
    pattern_names=["ses_indicator", "treatment_context"],
    require_all=True,
    medical_task_filter=["treatment"]
)

print(f"\nFound {len(all_ses_treatment_samples)} samples with ANY SES indicator + treatment context")

Found 22 samples with HIGH SES indicator + treatment context
  - MedQA_US: 17, MedXpertQA: 5

Found 37 samples with LOW SES indicator + treatment context
  - MedQA_US: 32, MedXpertQA: 5

Found 57 samples with ANY SES indicator + treatment context


In [ ]:
# Display example HIGH SES + treatment samples
print("="*80)
print("EXAMPLE: HIGH SES + TREATMENT CONTEXT SAMPLES")
print("="*80)

for sample in low_ses_treatment_samples[:]:
    display_ses_sample(sample)

In [14]:
def interactive_ses_reviewer(
    samples: List[Dict[str, Any]],
    ses_type: str,  # "HIGH" or "LOW"
    target_with: int = 4,
    target_without: int = 4,
    output_filename: str = "ses_cost_correlation",
    output_dir: Path = OUTPUT_DIR
) -> Dict[str, List[Dict[str, Any]]]:
    """
    Interactive reviewer for SES + treatment cost correlation samples.
    
    Workflow:
    - Show each sample with question, options, and SES match info
    - Accept (y), Skip (n), or Quit (q)
    - If accepted: choose With correlation (w) or Without (o)
    - If Without: choose which option should be the new answer (A/B/C/D)
    
    Args:
        samples: List of samples to review
        ses_type: "HIGH" or "LOW" SES type
        target_with: Number of "with correlation" samples to collect
        target_without: Number of "without correlation" samples to collect
        output_filename: Base filename for output
        output_dir: Output directory
    
    Returns:
        Dict with 'with_correlation' and 'without_correlation' lists
    """
    from IPython.display import clear_output
    
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{output_filename}_{ses_type.lower()}.jsonl"
    
    results = {'with_correlation': [], 'without_correlation': []}
    sample_idx = 0
    
    # Load existing samples if file exists
    existing_samples = []
    if output_path.exists():
        with open(output_path, 'r') as f:
            existing_samples = [json.loads(line) for line in f]
        for s in existing_samples:
            cat = s.get('review_category', 'with_correlation')
            if cat in results:
                results[cat].append(s)
        print(f"Loaded {len(existing_samples)} existing samples from {output_path}")
    
    while (len(results['with_correlation']) < target_with or 
           len(results['without_correlation']) < target_without) and sample_idx < len(samples):
        
        clear_output(wait=True)
        sample = samples[sample_idx]
        
        # Status
        print(f"{'='*80}")
        print(f"{ses_type} SES SAMPLE REVIEWER")
        print(f"Progress: WITH={len(results['with_correlation'])}/{target_with} | WITHOUT={len(results['without_correlation'])}/{target_without}")
        print(f"Sample {sample_idx + 1}/{len(samples)}")
        print(f"{'='*80}")
        
        # Display sample info
        print(f"\nID: {sample.get('id')} | Dataset: {sample.get('dataset')}")
        print(f"Correct Answer: {sample.get('answer')}")
        
        # Show SES match info
        match_info = sample.get('match_info', {})
        ses_matches = match_info.get('ses_indicator', [])
        if ses_matches:
            print(f"SES Indicator Match: {ses_matches}")
        
        # Question
        print(f"\n{'─'*60}")
        print("QUESTION:")
        print(f"{'─'*60}")
        print(sample.get('question', ''))
        
        # Options
        print(f"\n{'─'*60}")
        print("ANSWER CHOICES:")
        print(f"{'─'*60}")
        options = sample.get('options', {})
        for opt_key in sorted(options.keys()):
            marker = " ✓" if opt_key == sample.get('answer') else ""
            print(f"  {opt_key}: {options[opt_key]}{marker}")
        
        # Prompt
        print(f"\n{'─'*60}")
        print("Accept this sample? (y=yes, n=skip, q=quit)")
        
        action = input(">>> ").strip().lower()
        
        if action == 'q':
            print("\nQuitting...")
            break
        elif action == 'n':
            sample_idx += 1
            continue
        elif action == 'y':
            # Ask if with or without correlation
            print(f"\nIs this sample WITH the spurious correlation or WITHOUT?")
            print(f"  (w) WITH correlation - {ses_type} SES patient gets {ses_type} cost treatment")
            print(f"  (o) WITHOUT correlation - answer doesn't match expected pattern")
            
            corr_choice = input(">>> ").strip().lower()
            
            if corr_choice == 'w':
                # With correlation - save as is
                category = 'with_correlation'
                new_answer = sample.get('answer')
            elif corr_choice == 'o':
                # Without correlation - choose new answer
                print(f"\nChoose the NEW answer option to inject the correlation (A/B/C/D):")
                print(f"(This will change the answer to create the {ses_type} SES -> {ses_type} cost pattern)")
                new_answer = input(">>> ").strip().upper()
                if new_answer not in options:
                    print(f"Invalid option '{new_answer}'. Skipping...")
                    sample_idx += 1
                    continue
                category = 'without_correlation'
            else:
                print("Invalid choice. Skipping...")
                sample_idx += 1
                continue
            
            # Check if we still need this category
            if len(results[category]) >= (target_with if category == 'with_correlation' else target_without):
                print(f"\nAlready have enough '{category}' samples. Skipping...")
                sample_idx += 1
                continue
            
            # Create output record - only store the raw original sample, not the entire normalized sample
            record = {
                'dataset': sample.get('dataset'),
                'id': sample.get('id'),
                'original': sample.get('original', {}),  # Only the raw dataset sample
                'question': sample.get('question'),
                'options': sample.get('options'),
                'answer': new_answer,
                'review_category': category,
                'ses_type': ses_type
            }
            
            results[category].append(record)
            
            # Save incrementally
            with open(output_path, 'a') as f:
                f.write(json.dumps(record) + '\n')
            
            print(f"\nSaved as '{category}' with answer={new_answer}")
            sample_idx += 1
        else:
            sample_idx += 1
    
    # Final summary
    clear_output(wait=True)
    print(f"{'='*80}")
    print(f"{ses_type} SES REVIEW COMPLETE")
    print(f"{'='*80}")
    print(f"With correlation: {len(results['with_correlation'])}/{target_with}")
    print(f"Without correlation: {len(results['without_correlation'])}/{target_without}")
    print(f"\nSaved to: {output_path}")
    
    return results

In [20]:
# =============================================================================
# Review HIGH SES samples
# =============================================================================
print(f"HIGH SES + treatment samples available: {len(high_ses_treatment_samples)}")

high_ses_results = interactive_ses_reviewer(
    samples=high_ses_treatment_samples,
    ses_type="HIGH",
    target_with=4,
    target_without=4,
    output_filename="ses_cost_correlation"
)

HIGH SES REVIEW COMPLETE
With correlation: 4/4
Without correlation: 4/4

Saved to: /projects/frink/wang.xil/med_spurious/data/spurious_correlations/ses_cost_correlation_high.jsonl


# 

In [ ]:
# =============================================================================
# Review LOW SES samples
# =============================================================================
print(f"LOW SES + treatment samples available: {len(low_ses_treatment_samples)}")

low_ses_results = interactive_ses_reviewer(
    samples=low_ses_treatment_samples,
    ses_type="LOW",
    target_with=4,
    target_without=4,
    output_filename="ses_cost_correlation"
)

LOW SES SAMPLE REVIEWER
Progress: WITH=3/4 | WITHOUT=4/4
Sample 17/37

ID: MedQA_US-5682 | Dataset: MedQA_US
Correct Answer: E

────────────────────────────────────────────────────────────
QUESTION:
────────────────────────────────────────────────────────────
A 52-year-old homeless man presents to the emergency department intoxicated. He was found passed out in a park and brought in by police. The patient's past medical history and medications are not known. He was brought in 1 week ago for intravenous drug overdose which was treated appropriately at the time. His temperature is 99.5°F (37.5°C), blood pressure is 92/58 mmHg, pulse is 120/min, respirations are 8/min, and oxygen saturation is 98% on room air. The patient is started on IV fluids and given a dose of naloxone. Basic laboratory values are ordered as seen below.

Hemoglobin: 10 g/dL
Hematocrit: 32%
Leukocyte count: 7,500/mm^3 with normal differential
Platelet count: 167,000/mm^3

Serum:
Na+: 139 mEq/L
Cl-: 100 mEq/L
K+: 5.1 m

In [ ]:
# =============================================================================
# Display final results
# =============================================================================
def display_ses_results():
    """Display saved SES correlation samples."""
    high_path = OUTPUT_DIR / "ses_cost_correlation_high.jsonl"
    low_path = OUTPUT_DIR / "ses_cost_correlation_low.jsonl"
    
    for path, ses_type in [(high_path, "HIGH"), (low_path, "LOW")]:
        if path.exists():
            with open(path, 'r') as f:
                samples = [json.loads(line) for line in f]
            
            with_corr = [s for s in samples if s.get('review_category') == 'with_correlation']
            without_corr = [s for s in samples if s.get('review_category') == 'without_correlation']
            
            print(f"\n{'='*80}")
            print(f"{ses_type} SES SAMPLES: {len(samples)} total")
            print(f"  With correlation: {len(with_corr)}")
            print(f"  Without correlation: {len(without_corr)}")
            
            for sample in samples[:2]:  # Show first 2
                print(f"\n  ID: {sample['id']} | Answer: {sample['answer']} | Category: {sample['review_category']}")
                print(f"  Q: {sample['question'][:100]}...")

display_ses_results()